In [1]:
!pip install "datasets<3.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
from huggingface_hub import login
from google.colab import userdata

login(userdata.get('hf_read_token'))

In [3]:
from datasets import load_dataset
import os

# Create directory if it doesn't exist
output_dir = "./content/drive/MyDrive/stage 2 datasets 2/l3cube-pune"
os.makedirs(output_dir, exist_ok=True)

print("Downloading L3Cube-MahaSum dataset...")
# Load the dataset
mahasum_dataset = load_dataset("l3cube-pune/Marathi-Summarization")

# Save locally for offline training
mahasum_dataset.save_to_disk(output_dir)

print(f"Successfully saved MahaSum to {output_dir}")
# To load offline later: load_from_disk("./offline_datasets/mahasum")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetNotFoundError: Dataset 'l3cube-pune/Marathi-Summarization' doesn't exist on the Hub or cannot be accessed.

In [19]:
from datasets import load_dataset
import os

output_dir = './drive/MyDrive/stage_2_datasets_2/xlsum_marathi'
os.makedirs(output_dir, exist_ok=True)

print("Downloading XL-Sum (Marathi subset)...")

# Added trust_remote_code=True to bypass the script restriction
xlsum_dataset = load_dataset("csebuetnlp/xlsum", "marathi", trust_remote_code=True)

xlsum_dataset.save_to_disk(output_dir)
print(f"Successfully saved XL-Sum to {output_dir}")

Saving the dataset (0/1 shards):   0%|          | 0/10903 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1362 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1362 [00:00<?, ? examples/s]

Successfully saved XL-Sum to ./drive/MyDrive/stage_2_datasets_2/xlsum_marathi


In [20]:
from datasets import load_dataset
import os

output_dir = "./drive/MyDrive/stage_2_datasets_2/indic_marathi"
os.makedirs(output_dir, exist_ok=True)

print("Downloading AI4Bharat Indic Summarization (Marathi subset)...")
# 'mr' is the correct configuration for Marathi here
indic_dataset = load_dataset("ai4bharat/IndicSentenceSummarization", "mr", trust_remote_code=True)

indic_dataset.save_to_disk(output_dir)
print(f"Successfully saved AI4Bharat Summarization to {output_dir}")

Saving the dataset (0/1 shards):   0%|          | 0/86574 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10857 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10827 [00:00<?, ? examples/s]

Successfully saved AI4Bharat Summarization to ./drive/MyDrive/stage_2_datasets_2/indic_marathi


In [21]:
import os
from datasets import load_from_disk
from google.colab import drive

# 2. Define your source directories (where the train/test/validation folders live)
xlsum_base_dir = './drive/MyDrive/stage_2_datasets_2/xlsum_marathi'
indic_base_dir = './drive/MyDrive/stage_2_datasets_2/indic_marathi'

# 3. Define where you want to save the new CSV files
csv_output_base = './drive/MyDrive/stage_2_dataset_2_csv'

# List of the standard splits you found in your folders
splits = ['train', 'test', 'validation']

# ==========================================
# CONVERT XL-SUM
# ==========================================
print("--- Processing XL-Sum Dataset ---")
xlsum_csv_dir = os.path.join(csv_output_base, 'xlsum_marathi')
os.makedirs(xlsum_csv_dir, exist_ok=True)

for split in splits:
    # Point directly to the folder containing data-00000-of-00001.arrow
    split_folder = os.path.join(xlsum_base_dir, split)

    if os.path.exists(split_folder):
        print(f"Loading XL-Sum {split} split from: {split_folder}")
        try:
            # Load the specific subfolder directly
            dataset_split = load_from_disk(split_folder)

            # Define output CSV path
            csv_file_path = os.path.join(xlsum_csv_dir, f"{split}.csv")

            # Convert to CSV
            print(f"  -> Exporting to CSV: {csv_file_path}")
            dataset_split.to_csv(csv_file_path, index=False)
        except Exception as e:
            print(f"  ❌ Error processing {split}: {e}")
    else:
        print(f"  ⚠️ Split folder '{split}' not found in {xlsum_base_dir}")

print("\n" + "="*40 + "\n")

# ==========================================
# CONVERT AI4BHARAT (INDIC SUM)
# ==========================================
print("--- Processing Indic Summarization Dataset ---")
indic_csv_dir = os.path.join(csv_output_base, 'indic_marathi')
os.makedirs(indic_csv_dir, exist_ok=True)

for split in splits:
    split_folder = os.path.join(indic_base_dir, split)

    if os.path.exists(split_folder):
        print(f"Loading Indic Sum {split} split from: {split_folder}")
        try:
            dataset_split = load_from_disk(split_folder)
            csv_file_path = os.path.join(indic_csv_dir, f'{split}.csv')

            print(f"  -> Exporting to CSV: {csv_file_path}")
            dataset_split.to_csv(csv_file_path, index=False)
        except Exception as e:
            print(f"  ❌ Error processing {split}: {e}")
    else:
        print(f"  ⚠️ Split folder '{split}' not found in {indic_base_dir}")

print("\n🎉 Conversion completed successfully!")

--- Processing XL-Sum Dataset ---
Loading XL-Sum train split from: ./drive/MyDrive/stage_2_datasets_2/xlsum_marathi/train
  -> Exporting to CSV: ./drive/MyDrive/stage_2_dataset_2_csv/xlsum_marathi/train.csv


Creating CSV from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

Loading XL-Sum test split from: ./drive/MyDrive/stage_2_datasets_2/xlsum_marathi/test
  -> Exporting to CSV: ./drive/MyDrive/stage_2_dataset_2_csv/xlsum_marathi/test.csv


Creating CSV from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Loading XL-Sum validation split from: ./drive/MyDrive/stage_2_datasets_2/xlsum_marathi/validation
  -> Exporting to CSV: ./drive/MyDrive/stage_2_dataset_2_csv/xlsum_marathi/validation.csv


...

Loading Indic Sum test split from: ./drive/MyDrive/stage_2_datasets_2/indic_marathi/test
  -> Exporting to CSV: ./drive/MyDrive/stage_2_dataset_2_csv/indic_marathi/test.csv


Creating CSV from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

Loading Indic Sum validation split from: ./drive/MyDrive/stage_2_datasets_2/indic_marathi/validation
  -> Exporting to CSV: ./drive/MyDrive/stage_2_dataset_2_csv/indic_marathi/validation.csv


Creating CSV from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]


🎉 Conversion completed successfully!


In [22]:
import os
import numpy as np
from datasets import load_from_disk

# 1. Define the exact absolute paths to your mounted Drive
xlsum_path = './drive/MyDrive/stage_2_datasets_2/xlsum_marathi'
indic_path = './drive/MyDrive/stage_2_datasets_2/indic_marathi'

def calculate_dataset_stats(dataset_path, text_col, summary_col, dataset_name):
    print(f"\n" + "="*50)
    print(f"📊 Calculating Statistics for: {dataset_name}")
    print("="*50)

    if not os.path.exists(dataset_path):
        print(f"⚠️ Folder not found at {dataset_path}. Please check the spelling of your folders in Drive.\n")
        return

    print("Loading dataset into memory...")
    dataset = load_from_disk(dataset_path)

    text_lengths = []
    summary_lengths = []
    ratios = []

    # Loop through all splits (train, test, validation)
    for split in dataset.keys():
        print(f"  -> Processing '{split}' split...")
        for row in dataset[split]:

            # Extract text, handling any accidental empty values
            text = str(row[text_col]) if row[text_col] else ""
            summary = str(row[summary_col]) if row[summary_col] else ""

            # Calculate word count (splitting by whitespace for Marathi)
            t_len = len(text.split())
            s_len = len(summary.split())

            # Avoid division by zero for empty texts
            if t_len > 0 and s_len > 0:
                ratio = (s_len / t_len) * 100

                text_lengths.append(t_len)
                summary_lengths.append(s_len)
                ratios.append(ratio)

    # Calculate final statistics using NumPy
    print("\n--- Final Results ---")
    print(f"Total Article/Summary Pairs Processed: {len(text_lengths):,}")

    print("\n📝 TEXT LENGTH (Words):")
    print(f"  - Average (Mean): {np.mean(text_lengths):.1f}")
    print(f"  - Median:         {np.median(text_lengths):.1f}")
    print(f"  - Minimum:        {np.min(text_lengths)}")
    print(f"  - Maximum:        {np.max(text_lengths)}")

    print("\n📌 SUMMARY LENGTH (Words):")
    print(f"  - Average (Mean): {np.mean(summary_lengths):.1f}")
    print(f"  - Median:         {np.median(summary_lengths):.1f}")
    print(f"  - Minimum:        {np.min(summary_lengths)}")
    print(f"  - Maximum:        {np.max(summary_lengths)}")

    print("\n🗜️ COMPRESSION RATIO (Summary length / Text length):")
    print(f"  - Average Ratio:  {np.mean(ratios):.2f}%")
    print(f"  - Median Ratio:   {np.median(ratios):.2f}%")
    print("\n")

# ==========================================
# RUN THE CALCULATIONS
# ==========================================

# Run for XL-Sum (Columns: 'text' & 'summary')
calculate_dataset_stats(
    dataset_path=xlsum_path,
    text_col='text',
    summary_col='summary',
    dataset_name='XL-Sum (Marathi)'
)

# Run for Indic Summarization (Columns: 'input' & 'target')
calculate_dataset_stats(
    dataset_path=indic_path,
    text_col='input',
    summary_col='target',
    dataset_name='AI4Bharat Indic Summarization'
)


📊 Calculating Statistics for: XL-Sum (Marathi)
Loading dataset into memory...
  -> Processing 'train' split...
  -> Processing 'test' split...
  -> Processing 'validation' split...

--- Final Results ---
Total Article/Summary Pairs Processed: 13,627

📝 TEXT LENGTH (Words):
  - Average (Mean): 627.6
  - Median:         575.0
  - Minimum:        15
  - Maximum:        10308

📌 SUMMARY LENGTH (Words):
  - Average (Mean): 25.1
  - Median:         24.0
  - Minimum:        1
  - Maximum:        145

🗜️ COMPRESSION RATIO (Summary length / Text length):
  - Average Ratio:  7.27%
  - Median Ratio:   4.31%



📊 Calculating Statistics for: AI4Bharat Indic Summarization
Loading dataset into memory...
  -> Processing 'train' split...
  -> Processing 'test' split...
  -> Processing 'validation' split...

--- Final Results ---
Total Article/Summary Pairs Processed: 108,258

📝 TEXT LENGTH (Words):
  - Average (Mean): 19.6
  - Median:         18.0
  - Minimum:        4
  - Maximum:        95

📌 SUMMAR

In [25]:
import os
from datasets import load_from_disk

# 1. Define the exact absolute paths to your mounted Drive
xlsum_path = '/content/drive/MyDrive/stage_2_datasets_2/xlsum_marathi'
indic_path = '/content/drive/MyDrive/stage_2_datasets_2/indic_marathi'

def get_split_lengths(dataset_path, dataset_name):
    print(f"\n" + "="*40)
    print(f"📁 Dataset: {dataset_name}")
    print("="*40)

    if not os.path.exists(dataset_path):
        print(f"⚠️ Folder not found at {dataset_path}. Please check your Drive.")
        return

    try:
        # Load the dataset structure from your offline folder
        dataset = load_from_disk(dataset_path)
        total_rows = 0

        # Loop through whatever splits exist (train, test, validation)
        for split_name in dataset.keys():
            # len() instantly returns the number of rows in that split
            split_len = len(dataset[split_name])
            total_rows += split_len

            # Print the result nicely formatted with commas
            print(f"  - {split_name.capitalize():<12} : {split_len:,} rows")

        print("-" * 40)
        print(f"  - {'TOTAL':<12} : {total_rows:,} rows\n")

    except Exception as e:
        print(f"❌ Error loading dataset: {e}")

# ==========================================
# RUN THE COUNTER
# ==========================================

get_split_lengths(xlsum_path, "XL-Sum (Marathi)")
get_split_lengths(indic_path, 'AI4Bharat Indic Summarization')


📁 Dataset: XL-Sum (Marathi)
  - Train        : 10,903 rows
  - Test         : 1,362 rows
  - Validation   : 1,362 rows
----------------------------------------
  - TOTAL        : 13,627 rows


📁 Dataset: AI4Bharat Indic Summarization
  - Train        : 86,574 rows
  - Test         : 10,857 rows
  - Validation   : 10,827 rows
----------------------------------------
  - TOTAL        : 108,258 rows

